In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# 1. 위치 정보 포함 임베딩 (인코더와 동일한 공통 부품)
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

# 2. 디코더 블록 (미래를 보지 못하게 가리는 'Causal Mask'가 핵심)
def transformer_decoder_block(embed_dim, num_heads, ff_dim, rate=0.1):
    inputs = layers.Input(shape=(None, embed_dim))
    
    # [핵심] use_causal_mask=True : 현재 시점보다 뒤에 있는 단어는 가림
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim
    )(inputs, inputs, use_causal_mask=True)
    
    attn_output = layers.Dropout(rate)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attn_output)
    
    ffn_output = layers.Dense(ff_dim, activation="relu")(out1)
    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(rate)(ffn_output)
    
    return tf.keras.Model(inputs=inputs, outputs=layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output))

# 3. 데이터 준비 (학습을 위해 문장을 한 칸씩 밀어서 정답지 생성)
vocab_size, maxlen, embed_dim = 2000, 20, 32

# 학습용 샘플 문장 (반복 학습을 위해 양을 늘림)
sentences = [
    "장비 가동 상태 정상 유지",
    "현재 온도 수치 매우 안정",
    "모터 진동 발생 점검 필요",
    "긴급 상황 즉시 전원 차단"
] * 100 

vectorize_layer = layers.TextVectorization(max_tokens=vocab_size, output_sequence_length=maxlen)
vectorize_layer.adapt(sentences)

# 입력 데이터(X)와 정답 데이터(Y) 준비
# 디코더 학습의 핵심: "A B C"를 보고 "B C D"를 맞히도록 구성
text_ds = vectorize_layer(sentences)
X_train = text_ds[:, :-1] # 마지막 단어 제외
y_train = text_ds[:, 1:]  # 첫 번째 단어 제외 (한 칸씩 밀린 정답)

# 4. 디코더 모델 조립
inputs = layers.Input(shape=(maxlen-1,))
x = TokenAndPositionEmbedding(maxlen-1, vocab_size, embed_dim)(inputs)
x = transformer_decoder_block(embed_dim, 2, 32)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

decoder_model = tf.keras.Model(inputs=inputs, outputs=outputs)
decoder_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

print("--- 디코더 학습 시작 ---")
decoder_model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)
print("--- 학습 완료 ---\n")

# 5. 문장 생성 루프 (Inference Loop)
def generate_text(seed_text, max_gen_len=5):
    vocab = vectorize_layer.get_vocabulary()
    # 시작 단어를 숫자로 변환
    input_tokens = np.zeros((1, maxlen-1))
    initial_tokens = vectorize_layer([seed_text]).numpy()[0]
    
    # 첫 단어 채우기
    current_len = 0
    for val in initial_tokens:
        if val != 0:
            input_tokens[0, current_len] = val
            current_len += 1
            
    result = seed_text
    print(f"🌱 시작: {seed_text}")

    for i in range(current_len - 1, current_len - 1 + max_gen_len):
        # 다음에 올 모든 단어의 확률 예측
        predictions = decoder_model.predict(input_tokens, verbose=0)
        
        # 현재 시점(i)에서 가장 확률 높은 단어 선택
        next_token_id = np.argmax(predictions[0, i, :])
        next_word = vocab[next_token_id]
        
        if next_word == "" or next_word == "[UNK]": break
            
        result += " " + next_word
        print(f"   ㄴ Step {i - current_len + 2}: {result}")
        
        # 다음 루프를 위해 예측한 단어를 입력창에 추가
        if i + 1 < maxlen - 1:
            input_tokens[0, i + 1] = next_token_id
            
    return result




--- 디코더 학습 시작 ---
--- 학습 완료 ---

🌱 시작: 장비
   ㄴ Step 1: 장비 가동
   ㄴ Step 2: 장비 가동 상태
   ㄴ Step 3: 장비 가동 상태 정상
   ㄴ Step 4: 장비 가동 상태 정상 유지


'장비 가동 상태 정상 유지'

In [3]:
# 테스트 실행
generate_text("장비")
generate_text("온도")
generate_text("날씨")

🌱 시작: 장비
   ㄴ Step 1: 장비 가동
   ㄴ Step 2: 장비 가동 상태
   ㄴ Step 3: 장비 가동 상태 정상
   ㄴ Step 4: 장비 가동 상태 정상 유지
🌱 시작: 온도
   ㄴ Step 1: 온도 수치
   ㄴ Step 2: 온도 수치 매우
   ㄴ Step 3: 온도 수치 매우 안정
🌱 시작: 날씨


'날씨'